In [2]:
pip install pillow imagehash

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from PIL import Image
import imagehash

In [4]:


def remove_near_duplicates_classwise(dataset, threshold=5):
    total_removed = 0

    for class_name in ['real', 'fake']:
        class_path = os.path.join(dataset, class_name)
        hashes = {}
        removed = 0

        print(f"\n🔍 Checking {class_name.lower()} images...")

        for file in os.listdir(class_path):
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                path = os.path.join(class_path, file)

                try:
                    img = Image.open(path).convert('RGB')
                    img_hash = imagehash.phash(img)

                    duplicate_found = False
                    for h in hashes:
                        if abs(img_hash - h) <= threshold:
                            os.remove(path)
                            removed += 1
                            duplicate_found = True
                            break

                    if not duplicate_found:
                        hashes[img_hash] = path

                except Exception:
                    os.remove(path)
                    removed += 1

        print(f"✅ Removed {removed} images from {class_name}")
        total_removed += removed

    print(f"\n🎯 TOTAL images removed: {total_removed}")

# Run from Jupyter
remove_near_duplicates_classwise("dataset", threshold=5)



🔍 Checking real images...
✅ Removed 0 images from real

🔍 Checking fake images...
✅ Removed 274 images from fake

🎯 TOTAL images removed: 274


In [5]:
!python -m pip install opencv-python numpy matplotlib scikit-image scikit-learn


In [6]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed


In [7]:
IMG_SIZE = 128   # fixed size
BASE_PATH = r"C:\Users\djdes\OneDrive\Documents\Image Detection\dataset"

X = []   # features
Y = []   # labels

def preprocess_and_extract(img_path):
    # Read image
    img = cv2.imread(img_path)
    
    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Noise removal
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Edge detection
    edges = cv2.Canny(gray, 100, 200)
    
    # HOG feature extraction
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )
    
    return hog_features


# Load dataset with Parallel Processing
all_image_paths = []
all_labels = []

for label, folder in enumerate(['real', 'fake']):
    folder_path = os.path.join(BASE_PATH, folder)
    if not os.path.exists(folder_path): continue
    
    for file in os.listdir(folder_path):
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_image_paths.append(os.path.join(folder_path, file))
            all_labels.append(label)

print(f"Processing {len(all_image_paths)} images using parallel processing...")

def process_helper(path):
    try:
        return preprocess_and_extract(path)
    except:
        return None

results = Parallel(n_jobs=-1)(delayed(process_helper)(path) for path in all_image_paths)

# Filter results
for i, res in enumerate(results):
    if res is not None:
        X.append(res)
        Y.append(all_labels[i])


# Convert to numpy arrays
X = np.array(X)
Y = np.array(Y)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("Feature shape:", X.shape)
print("Label shape:", Y.shape)


Processing 19724 images using parallel processing...
Feature shape: (19724, 8100)
Label shape: (19724,)


In [8]:
print(os.listdir(BASE_PATH))


['fake', 'real']


In [9]:
#module 3
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 15779
Testing samples: 3945


In [11]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.001],
    'kernel': ['rbf']
}

svm = SVC()

grid = GridSearchCV(svm, param_grid, cv=3, verbose=2, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

model = grid.best_estimator_


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


In [18]:
import os
import cv2
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [19]:
def evaluate_model(model, X_test, y_test):
    print("--- EVALUATION ---")
    y_pred = model.predict(X_test)
    print("Accuracy Score:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['real', 'fake']))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return y_pred

In [20]:
def save_artifacts(model, scaler):
    print("\n--- SAVING ARTIFACTS ---")
    joblib.dump(model, 'best_svm_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')
    print("Model and Scaler saved successfully as 'best_svm_model.pkl' and 'scaler.pkl'")

In [21]:
def predict_single_image(img_path, model, scaler, preprocess_fn):
    # Preprocess and extract features
    features = preprocess_fn(img_path)
    
    if features is None:
        return "Error: Could not process image."
    
    # Reshape and scale
    features = features.reshape(1, -1)
    features = scaler.transform(features)
    
    # Predict
    prediction = model.predict(features)
    return "Fake" if prediction[0] == 1 else "Real"

In [22]:
import matplotlib.pyplot as plt
import random

def plot_prediction_gallery(X_test, y_test, y_pred, n_images=10):
    print(f"\n--- VISUALIZING {n_images} PREDICTIONS ---")
    plt.figure(figsize=(15, 6))
    
    # Randomly pick indices
    indices = random.sample(range(len(X_test)), n_images)
    
    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i + 1)
        
        # Note: SVM was trained on HOG features, not raw pixels.
        # So we can't easily plot the image from X_test directly.
        # In the notebook, you should use the original 'all_image_paths' list.
        
        plt.text(0.5, 0.5, f"Pred: {'Fake' if y_pred[idx]==1 else 'Real'}\nActual: {'Fake' if y_test[idx]==1 else 'Real'}", 
                 horizontalalignment='center', verticalalignment='center',
                 color='green' if y_pred[idx] == y_test[idx] else 'red',
                 fontsize=12, fontweight='bold')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()